In [1]:
from dbfread import DBF
import polars as pl
import os
from read_enoe import *

In [ ]:
# Ruta donde guardarás los archivos .parquet
output_dir = "/Users/mariajosecota/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/LSE/CAPSTONE/my498-capstone/data/enoe_parquets"

# Trimestres y años a procesar
# SINCE 2005 to 2019
years = [2010, 2011, 2012, 2013, 2014, 2013, 2015, 2016, 2017, 2018, 2019]
trimesters =  [2, 3, 4]

# Diccionario opcional de columnas a conservar por módulo (puedes ajustarlo)
columns_to_keep = { 
    "coe1": ["CD_A", "ENT", "CON", "V_SEL", "N_HOG", "H_MUD", "N_REN",
             "PR3_ANIO", "P3R_MES"],
    "coe2": ["CD_A", "ENT", "CON", "V_SEL", "N_HOG", "H_MUD", "N_REN",
              "P9_2", "P9_3", "P9_4", "P9_5", "P9_6", "P9_7", "P9_8", 
              "P9_H2", "P9_H3", "P9_H4", "P9_H5", "P9_H6", "P9_H7", "P9_H8",
              "P9_M2", "P9_M3", "P9_M4", "P9_M5", "P9_M6", "P9_M7", "P9_M8"],
    "hogt": ["CD_A", "ENT", "CON", "V_SEL", "UPM", "N_PRO_VIV", "N_HOG", "H_MUD", "R_PRE", "UR", "P4_1",
             "D_DIA", "D_MES", "D_ANIO"],
    "vivt": ["CD_A", "ENT", "CON", "V_SEL", "UPM", "N_PRO_VIV"],
    "sdemt": ["CD_A", "ENT", "CON", "V_SEL", "N_HOG", "H_MUD", "C_RES", "N_REN",
              "SEX", "EDA", "E_CON", "PAR_C", "NIV_INS", "PER", "N_HIJ",
              "CLASE1", "CLASE2", "HRSOCUP", "INGOCUP", "ING_X_HRS",
              "EMP_PPAL", "T_TRA", "ANIOS_ESC", "P14APOYOS", "SCIAN", "T_TRA", "EMP_PPAL", "TUE_PPAL"
              ]
}

# Base donde están los archivos .dbf
base_dir = "/Users/mariajosecota/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/LSE/CAPSTONE/my498-capstone/data/enoe"

for year in years:
    for trimester in trimesters:
        folder_name, _ = generate_keys(year, trimester)
        parquet_path = os.path.join(output_dir, f"{folder_name}.parquet")

        if os.path.exists(parquet_path):
            print(f"✅ Ya existe: {parquet_path} — se omite.")
            continue

        try:
            print(f"⏳ Procesando {year}T{trimester}...")
            df = read_and_merge_enoe(year, trimester, base_dir, selected_columns=columns_to_keep)
            df.write_parquet(parquet_path)
            print(f"✅ Guardado: {parquet_path}\n")
        except Exception as e:
            print(f"❌ Error en {year}T{trimester}: {e}\n")

        

✅ Ya existe: /Users/mariajosecota/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/LSE/CAPSTONE/my498-capstone/data/enoe_parquets/2010trim2.parquet — se omite.
✅ Ya existe: /Users/mariajosecota/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/LSE/CAPSTONE/my498-capstone/data/enoe_parquets/2010trim3.parquet — se omite.
✅ Ya existe: /Users/mariajosecota/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/LSE/CAPSTONE/my498-capstone/data/enoe_parquets/2010trim4.parquet — se omite.
✅ Ya existe: /Users/mariajosecota/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/LSE/CAPSTONE/my498-capstone/data/enoe_parquets/2011trim2.parquet — se omite.
✅ Ya existe: /Users/mariajosecota/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/LSE/CAPSTONE/my498-capstone/data/enoe_parquets/2011trim3.parquet — se omite.
✅ Ya existe: /Users/mariajosecota/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/LSE/CAPSTONE/my498-capstone/data/enoe_parquets/2011trim4.parquet — se omite.
✅ Ya exist

In [4]:
import polars as pl
import os

# Ruta donde están los archivos .parquet por trimestre
parquet_dir = "/Users/mariajosecota/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/LSE/CAPSTONE/my498-capstone/data/enoe_parquets"

# Listar archivos válidos que tienen año y trimestre en el nombre
parquet_files = [
    os.path.join(parquet_dir, f)
    for f in os.listdir(parquet_dir)
    if f.endswith(".parquet") and "trim" in f and f[:4].isdigit()
]

# Función para extraer año y trimestre desde el nombre del archivo
def extract_year_trim(filepath):
    basename = os.path.basename(filepath)  # ej. "2016trim1.parquet"
    
    if not ("trim" in basename and basename[:4].isdigit()): 
        raise ValueError(f"Nombre de archivo inesperado: {basename}")

    year = int(basename[:4])
    trimester = int(basename.split("trim")[1].split(".")[0])
    return pl.lit(year).alias("year"), pl.lit(trimester).alias("trimester")

# Leer cada archivo, agregar columnas year y trimester, y almacenar en lista
dfs = []
for file in parquet_files:
    df = pl.read_parquet(file)
    year_col, trim_col = extract_year_trim(file)
    df = df.with_columns([year_col, trim_col])
    dfs.append(df)

# Concatenar todos los DataFrames en uno solo
df_all = pl.concat(dfs, how="vertical")

# Guardar el archivo unificado
output_path = os.path.join(parquet_dir, "enoe_all.parquet")
df_all.write_parquet(output_path)

print(f"✅ Archivo unido guardado como: {output_path}")

✅ Archivo unido guardado como: /Users/mariajosecota/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/LSE/CAPSTONE/my498-capstone/data/enoe_parquets/enoe_all.parquet


In [5]:
import polars as pl


file_path = "/Users/mariajosecota/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/LSE/CAPSTONE/my498-capstone/data/enoe_parquets/enoe_all.parquet"
df_all = pl.read_parquet(file_path)

# Count observations 
df_all.shape[0] #  4,404,563

8786531

In [6]:
# Display columns:     "D_DIA", "D_MES", "D_ANIO" ,"PR3_ANIO", "P3R_MES"
df_all.select([
    "D_DIA", "D_MES", "D_ANIO", "PR3_ANIO", "P3R_MES"
]).head(5)

ColumnNotFoundError: D_DIA

In [5]:
# Identify individuals in a union or marriage that live in the same household. Also determine the type of couple (M-MM, F-F, M-F)

# Step 1: Filter adults in a union or marriage 
couples_i = df_all.filter(
    (pl.col("E_CON").cast(pl.Int64).is_in([1, 5])) &     # Casado o unión libre
    (pl.col("PAR_C").cast(pl.Int64).is_in([101, 201])) & # Jefe o cónyuge
    (pl.col("EDA").cast(pl.Int64) >= 18)                 # Adultos
).with_columns([
    pl.lit(1).alias("is_couple")
])

# Step 2: Keep households with exactly one head and one spouse
target_hh = (
    couples_i
    .with_columns([
        pl.col("PAR_C").cast(pl.Int64)  # Asegura que sea numérica antes de agrupar
    ])
    .group_by(["hogar_id", "PER"])
    .agg([
        pl.len().alias("n"),
        (pl.col("PAR_C") == 101).sum().alias("n_heads"),
        (pl.col("PAR_C") == 201).sum().alias("n_spouses")
    ])
    .filter(
        (pl.col("n") == 2) &
        (pl.col("n_heads") == 1) &
        (pl.col("n_spouses") == 1)
    )
    .select("hogar_id", "PER")
)

# 3. Join to get the info about the characteristics of the couple
couples_i = (couples_i.join(
        target_hh,
        on=["hogar_id", "PER"],
        how="inner"
    ))

# Step 4: Identify couple type and Keep only target couples (Male-Female)
couples_i = couples_i.with_columns(pl.col("SEX").cast(pl.Int64))

couple_types = (
    couples_i
    .group_by("hogar_id", "PER")
    .agg([
        pl.col("SEX").first().alias("sex_1"),
        pl.col("SEX").last().alias("sex_2")
    ])
    .with_columns(
        pl.when((pl.col("sex_1") == 1) & (pl.col("sex_2") == 1))
        .then(pl.lit("Male-Male"))
        .when((pl.col("sex_1") == 2) & (pl.col("sex_2") == 2))
        .then(pl.lit("Female-Female"))
        .otherwise(pl.lit("Male-Female"))
        .alias("couple_type")
    )
)

# 3. Make Relevant Transformations Individuals

In [6]:
import polars as pl

# Diccionario de renombrado (como en R)
rename_dict = {
    "P9_2": "unpaid_care",
    "P9_3": "unpaid_shop",
    "P9_4": "transport",
    "P9_5": "build",
    "P9_6": "repair",
    "P9_7": "housework",
    "P9_8": "community",

    "P9_M2": "unpaid_care_minutes",
    "P9_M3": "unpaid_shop_minutes",
    "P9_M4": "transport_minutes",
    "P9_M5": "build_minutes",
    "P9_M6": "repair_minutes",
    "P9_M7": "housework_minutes",
    "P9_M8": "community_minutes",

    "P9_H2": "unpaid_care_hours",
    "P9_H3": "unpaid_shop_hours",
    "P9_H4": "transport_hours",
    "P9_H5": "build_hours",
    "P9_H6": "repair_hours",
    "P9_H7": "housework_hours",
    "P9_H8": "community_hours",

    "HRSOCUP": "weekly_paid_hours",
    "INGOCUP": "monthly_income",
    "ING_X_HRS": "income_per_hour",
    "SEX": "sex",
    "EDA": "age",

    "EMP_PPAL": "main_job_formality",             # 1 = informal, 2 = formal
    "TUE_PPAL": "main_job_sector",                # 1 = sector informal, 2 = fuera
    "T_TRA": "total_jobs",                        # 1 = un solo trabajo, 2 = dos trabajos
    "ANIOS_ESC": "years_of_education",            # 1–24 años, 99 = no especificado
    "P14APOYOS": "receives_financial_support",    # 1 = sí, 2 = no, 3 = no especificado
    "SCIAN": "economic_sector_code"               # 1–21 sectores según SCIAN
}

# List of unpaid labor vars
activity_vars = ["unpaid_care", "unpaid_shop", "transport", "build", "repair", "housework", "community"]
hours_vars = [f"{var}_hours" for var in activity_vars]
minutes_vars = [f"{var}_minutes" for var in activity_vars]

# Paso 1: Renombrar columnas
df_all = df_all.rename(rename_dict)

In [7]:
# Variables
activity_vars = ["unpaid_care", "unpaid_shop", "transport", "build", "repair", "housework", "community"]
hours_vars = [f"{var}_hours" for var in activity_vars]
minutes_vars = [f"{var}_minutes" for var in activity_vars]
time_vars = hours_vars + minutes_vars

# Recode 99 to missing value
df_all = df_all.with_columns(
    pl.when(pl.col("years_of_education") == 99)
    .then(None)
    .otherwise(pl.col("years_of_education"))
    .alias("years_of_education")
)

# Recoding. main_job_formality should be 1 if curret value is 2 (formal), and 0 if it is 1 (informal). Same for main_job_sector.
df_all = df_all.with_columns(
    pl.when(pl.col("main_job_formality") == 2)
    .then(1)
    .when(pl.col("main_job_formality") == 1)
    .then(0)
    .otherwise(None)  # Si no es 1 ni 2, asigna None
    .alias("main_job_formality")
).with_columns(
    pl.when(pl.col("main_job_sector") == 2)
    .then(1)
    .when(pl.col("main_job_sector") == 1)
    .then(0)
    .otherwise(None)  # Si no es 1 ni 2, asigna None
    .alias("main_job_sector")
)


# Limpiar "" y valores 98/99, y convertir a float
df_all = (
    df_all
    .with_columns([
        pl.when(pl.col(col).str.strip_chars() == "")
          .then(None)
          .otherwise(pl.col(col))
          .alias(col)
        for col in time_vars
    ])
    .with_columns([
        pl.when(pl.col(col).cast(pl.Float64).is_in([98.0, 99.0]))
          .then(None)
          .otherwise(pl.col(col).cast(pl.Float64))
          .alias(col)
        for col in time_vars
    ])
)

# Crear indicadores binarios por actividad
df_all = df_all.with_columns([
    pl.when(pl.col(var).is_null())
      .then(0)
      .otherwise(1)
      .cast(pl.Int8)
      .alias(var)
    for var in activity_vars
])

In [8]:
# C: unpaid_labor → al menos una actividad con valor 1
df_all = df_all.with_columns([
    (sum([pl.col(var) for var in activity_vars]) > 0)
    .cast(pl.Int8)
    .alias("unpaid_labor")
])

# D: total_unpaid_hours (suma de horas, NA = 0)
df_all = df_all.with_columns([
    sum([pl.col(var).fill_null(0) for var in hours_vars])
    .alias("total_unpaid_hours")
])

# E: total_unpaid_minutes (suma de minutos, NA = 0)
df_all = df_all.with_columns([
    sum([pl.col(var).fill_null(0) for var in minutes_vars])
    .alias("total_unpaid_minutes")
])

# F: weekly_unpaid_hours = horas + minutos/60
df_all = df_all.with_columns([
    (pl.col("total_unpaid_hours") + pl.col("total_unpaid_minutes") / 60)
    .alias("weekly_unpaid_hours")
])


# C: unpaid_labor → al menos una actividad con valor 1
df_all = df_all.with_columns([
    (sum([pl.col(var) for var in activity_vars]) > 0)
    .cast(pl.Int8)
    .alias("unpaid_labor")
])

# D: total_unpaid_hours (suma de horas, NA = 0)
df_all = df_all.with_columns([
    sum([pl.col(var).fill_null(0) for var in hours_vars])
    .alias("total_unpaid_hours")
])

# E: total_unpaid_minutes (suma de minutos, NA = 0)
df_all = df_all.with_columns([
    sum([pl.col(var).fill_null(0) for var in minutes_vars])
    .alias("total_unpaid_minutes")
])

# F: weekly_unpaid_hours = horas + minutos/60
df_all = df_all.with_columns([
    (pl.col("total_unpaid_hours") + pl.col("total_unpaid_minutes") / 60)
    .alias("weekly_unpaid_hours")
])

# F: weekly_unpaid_hours = horas + minutos/60
df_all = df_all.with_columns([
    (pl.col("total_unpaid_hours") + pl.col("total_unpaid_minutes") / 60)
    .alias("weekly_unpaid_hours")
])



# SPECIFICS
# Calcular weekly_unpaid_care_hours: horas + minutos/60, NA = 0
df_all = df_all.with_columns([
    (
        pl.col("unpaid_care_hours").fill_null(0) +
        pl.col("unpaid_care_minutes").fill_null(0) / 60
    ).alias("weekly_unpaid_care_hours")
])


In [9]:
df_all = df_all.with_columns([
    # Convertir todas las columnas a numérico antes de las operaciones
    pl.col("age").cast(pl.Int64),
    pl.col("sex").cast(pl.Int64)
]).with_columns([
    pl.when(pl.col("sex") == 1).then(pl.lit("Male"))
    .when(pl.col("sex") == 2).then(pl.lit("Female"))
    .otherwise(pl.lit("Other"))
    .alias("sex_label")
])

In [10]:
# HH INFO

In [11]:
# Get household information and aggregate by household and period


# 2. Unir con couple_types y agregar info por hogar y periodo
household_info = (
    df_all.join(  # Solo hogares con parejas
        couple_types,
        on=["hogar_id", "PER"],
        how="inner"
    )
    .with_columns([ 
        # Convertir todas las columnas a numérico antes de las operaciones
        (pl.col("UR").cast(pl.Int32) == 2).cast(pl.Int8).alias("urban_rural"),  
        pl.col("age").cast(pl.Int64).alias("age"), 
        pl.col("sex").cast(pl.Int64).alias("sex"), 
        (pl.col("P4_1").cast(pl.Int32).is_in([1, 2, 3])).cast(pl.Int8).alias("domestic_workers")
    ])
    
    .group_by("hogar_id", "PER")
    .agg([
        # Variables de contexto
        pl.col("urban_rural").first().alias("urban_rural"),
        pl.col("couple_type").first().alias("couple_type"),
        pl.col("domestic_workers").first().alias("domestic_workers"),  # Puedes cambiar esto según lo que represente

        # Total de personas en el hogar
        pl.len().alias("n_personas_totales"),
        
        # Ingreso del hogar - CONVERTIR A NUMÉRICO
        pl.col("monthly_income").cast(pl.Float64).sum().alias("monthly_hh_income"),

        # Total weekly unpaid hours
        pl.col("weekly_unpaid_hours").cast(pl.Float64).sum().alias("total_weekly_unpaid_hours_hh"),

        # Conteo de menores de 18
        pl.col("age").filter(pl.col("age") < 18).len().alias("n_menores_18"),

        # Conteo de menores de 12
        pl.col("age").filter(pl.col("age") <= 12).len().alias("n_menores_12"),

        # Conteo de mujeres (SEX == 2)
        pl.col("sex").filter(pl.col("sex") == 2).len().alias("n_mujeres"),

        # Conteo de hombres (SEX == 1)
        pl.col("sex").filter(pl.col("sex") == 1).len().alias("n_hombres")
    ])
)


# 4. Join

In [12]:
df_all_filtered = (
    df_all
    .join(couples_i, on=["hogar_id", "PER", "N_REN"], how="left")
    .join(household_info, on=["hogar_id", "PER"], how="inner")
    .with_columns([
        pl.col("is_couple").fill_null(0)  # Llenar nulos con 0
    ]) 
    .select(
        pl.col("sex"), pl.col("sex_label"),
        pl.col("N_REN"),
        pl.col("hogar_id"),
        pl.col("PER"),   
        pl.col("income_per_hour"),
        pl.col("monthly_income"), 
        pl.col("unpaid_labor"),
        pl.col("weekly_unpaid_hours"),
        pl.col("unpaid_care"),
        pl.col("weekly_unpaid_care_hours"),
        pl.col("weekly_paid_hours"),
        pl.col("age"),
        pl.col("years_of_education"),
        pl.col("receives_financial_support"),
        pl.col("economic_sector_code"),
        pl.col("main_job_formality"),
        pl.col("main_job_sector"),
        pl.col("total_jobs"),
        pl.col("couple_type"),
        pl.col("is_couple"),
        pl.col("PAR_C")
    )
    .filter(pl.col("couple_type") == "Male-Female")
)


In [13]:
df_hh_couple = (
    df_all_filtered
    # Filter is_couple == 1
    .filter(pl.col("is_couple") == 1)
    .pivot(
        values=[
            "income_per_hour",
            "monthly_income",
            "unpaid_labor",
            "unpaid_care",
            "weekly_unpaid_care_hours",
            "weekly_unpaid_hours",
            "weekly_paid_hours",
            "age", 
            "years_of_education",
            "receives_financial_support",
            "economic_sector_code",
            "main_job_formality",
            "main_job_sector",
            "total_jobs"
        ],
        index=["hogar_id", "PER", "is_couple"],
        columns="sex_label"
    )
)


/var/folders/pl/_zpty6zn7pzcs_9khc4f43xc0000gn/T/ipykernel_40714/2055213287.py:5: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(


In [14]:
appearances = df_hh_couple.group_by("hogar_id").agg(pl.len())
df_hh_couple = df_hh_couple.join(appearances, on="hogar_id", how="inner") # rename len column to "n_appearances"
df_hh_couple = df_hh_couple.rename({"len": "n_appearances"})
df_hh_couple = df_hh_couple.join(household_info, on=["hogar_id", "PER"], how="inner")

path = "/Users/mariajosecota/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/LSE/CAPSTONE/my498-capstone/data/enoe_parquets/final_panel.parquet"
df_hh_couple.write_parquet(path) 

In [15]:
# ver df_hh
df_hh_couple.head()

hogar_id,PER,is_couple,income_per_hour_Male,income_per_hour_Female,monthly_income_Male,monthly_income_Female,unpaid_labor_Male,unpaid_labor_Female,unpaid_care_Male,unpaid_care_Female,weekly_unpaid_care_hours_Male,weekly_unpaid_care_hours_Female,weekly_unpaid_hours_Male,weekly_unpaid_hours_Female,weekly_paid_hours_Male,weekly_paid_hours_Female,age_Male,age_Female,years_of_education_Male,years_of_education_Female,receives_financial_support_Male,receives_financial_support_Female,economic_sector_code_Male,economic_sector_code_Female,main_job_formality_Male,main_job_formality_Female,main_job_sector_Male,main_job_sector_Female,total_jobs_Male,total_jobs_Female,n_appearances,urban_rural,couple_type,domestic_workers,n_personas_totales,monthly_hh_income,total_weekly_unpaid_hours_hh,n_menores_18,n_menores_12,n_mujeres,n_hombres
str,str,i32,f64,f64,i64,i64,i8,i8,i8,i8,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i32,i32,i32,i32,i64,i64,u32,i8,str,i8,u32,f64,f64,u32,u32,u32,u32
"""0660015150660076018020""","""217""",1,21.42857,39.58333,6450,8170,1,1,1,1,0.0,14.0,0.0,38.0,70,48,39,39,6,11,0,0,1,5,0,1,1,1,1,1,3,1,"""Male-Female""",0,3,20640.0,38.0,0,0,1,2
"""2650056012601283008110""","""418""",1,0.0,0.0,0,0,1,1,1,1,0.0,0.0,7.0,31.0,46,45,44,46,7,9,0,0,8,5,1,1,1,1,1,1,3,1,"""Male-Female""",0,3,0.0,52.0,0,0,2,1
"""1501219040024250010810""","""410""",1,16.66667,0.0,3870,0,1,1,1,1,9.0,25.0,11.0,43.0,54,0,26,32,9,9,0,0,5,0,0,null,0,null,1,1,4,0,"""Male-Female""",0,2,3870.0,54.0,0,0,1,1
"""0700124040000148020710""","""211""",1,0.0,0.0,8000,0,1,1,1,1,0.0,3.0,5.0,37.0,0,0,33,33,9,9,0,0,5,0,1,null,1,null,1,1,4,0,"""Male-Female""",0,3,8000.0,45.0,1,0,1,2
"""1540378041515817008610""","""314""",1,0.0,0.0,0,0,1,1,1,1,0.0,20.0,14.0,53.0,54,0,54,34,12,12,0,0,12,0,1,null,1,null,1,1,4,0,"""Male-Female""",0,2,0.0,67.0,0,0,1,1
